# Master Thesis Martin Vingerhoets 

## Section 1: Introduction 

### Staggered Grid Spatial Discretization of 1D and 2D SWE  

Q1: What criteria are available to set the mesh width (i.e. the spatial resolution) in a non-linear transient wave propagation problem? Should a minimum number of grid points per wavelength or otherwise be used? How is the largest frequency obtained? How is the mesh width influence by the order of spatial discretization? Note that:

1. higher harmonics are generated (by the non-linear terms in the equation) and;
2. one wishes to preserve a minimal number of grid point per wave-length?
3. how is this question resolved in [TrixiShallowWater](https://github.com/trixi-framework/TrixiShallowWater.jl)? 

Q2: what are reference implementations of staggered grid methods 
1. see [staggered grid](https://ptsolvers.github.io/JustRelax.jl/dev/man/equations_discretization) in JustRelax.jl;

Q3: Currently the harmonics are split into sin and cos terms. Is switching to complex-valued arithmetic a feasible way to reduce overall computational cost? 

### Explicit, Implicit and Hybrid Transient Solver 

Q1: How should a [steady state solver](https://docs.sciml.ai/DiffEqDocs/stable/types/steady_state_types/) be applied to solve a transient problem? Test this first on the harmonic oscillator. 

Q2: What kind of IMEX solvers ([Split ODE Problems](https://docs.sciml.ai/DiffEqDocs/stable/types/split_ode_types/)) are available to simulate shallow water flow? What are the implicit and explicit solvers that are being combined? 

Q3: How should IMEX solvers be deployed such that all non-linear terms are treated explicitly and that merely a **linear** system needs to be solver at each time? 

Q4: How are the linear systems resulting from IMEX methods solved? Can the resulting linear system be solved using a Schur (or approximation thereof) complement method as e.g. in [shallowWaterFoam.C](https://www.openfoam.com/documentation/guides/latest/api/shallowWaterFoam_8C.html#details). 

Q5: does a linear model allow to estimate the time required to reach a steady state? 

### (Non-)Linear Time-Harmonic Solver 

Decompose time-dependent part into harmonics (base harmonic and higher order harmonic generated by the non-linear term in the equation).  

Q1: what non-linear terms are taken into account? How is the term $\| \mathbf{u} \| \mathbf{u}$ treated? How can this term be simplified without loosing too much of the physics involved?  

Q2: how can the sparse Jacobian be generated using [ForwardDiff](https://github.com/JuliaDiff/ForwardDiff.jl). How to use [sparsity dedection](https://docs.sciml.ai/NonlinearSolve/stable/tutorials/large_systems/#Approximate-Sparsity-Detection-and-Sparse-Jacobians)? See also [Sparsity handling in DifferentiationInterface](https://juliadiff.org/DifferentiationInterface.jl/DifferentiationInterface/stable/tutorials/advanced/#Sparsity). 

Q3: what is the structure of the Jacobian? How to split Jacobian into a linear part (independent of the Newton step) and an non-linear part (varrying with the  Newton step). A3: see draft text Martin (Domenico needs to read);  

Q4: how to store the Jacobian as a [block matrix](https://juliaarrays.github.io/BlockArrays.jl/stable/man/blockarrays/) rendering the block preconditioner easier to implement? See [conversation with Guillaume Dalle on slack](https://julialang.slack.com/archives/C6A044SQH/p1774212987002119). 

Q5: how to convert coupled first order system as a second order system, thus making it feasible to replace the ILU preconditioner by the AMG preconditioiner; 

Q6: how to apply a conformal transformation on the linear part of the Jacobian to obtain all eigenvalues on the same side of the imaginary axis? 

Q7: how to freeze the preconditioner over various iterations on the Newton iteration? 

### Newton-LU for both Transient and Time-Harmonic Solvers

<b>Advantages of LU:</b> Robust. Mainstream, and thus well documented. What about modern variatiants (e.g. Pardiso), recursive factorizations or GPU implementations?    

<b>Disadvantages of LU:</b>

1. the large number of fill-in elements, especially in case of two-dimensional discretization with large number of harmonics;
2. solving too accuratey (oversolving) especially at first Newton iterations, when still far from the final solution (cfr. forcing terms, sufficient descent directions);

<b>Pointers to Direct Solver that Exploit Sparsity</b>

1. the forward and backward substitution is implemented in <i>ldiv!</i>;
2. the sparse symmetric and positive definite case is implemented in <i>CHOLMOD</i>. See [this Discourse post](https://discourse.julialang.org/t/defining-a-preconditioner-for-iterativesolvers/86977/17);  

### Newton-Krylov for both Transient and Time-Harmonic Solvers

<b>Advantages of Krylov:</b> Avoid oversolving using [forcing_strategies](https://docs.sciml.ai/NonlinearSolve/stable/native/solvers/#forcing_strategies). 

<b>Disadvantages of Krylov:</b> Slow to converge in case that the eigenvalues of the Jacobian spread accross both sides of the imaginary axis unless properly treated (using a conformal mapping as in the case of the complex-shifted Laplace preconditioner). 

Q1: How does one ensure that the GMRES search space is allocated once only, accross all of the Newton iterations? LinearSolve.jl builds it once in its cache init and then future solves just reinit!. 

Q2: how does the GMRES solver converge? What is the influence of the restart value on the overall computational cost? Is switching to a short-term Krylov solver (BiCGSTAB, IDR(s) or friend borrowed from petsc.jl) a feasible way to reduce overall computational cost?  

Q3: how to specify the preconditioner in the example on the page [forcing strategies](https://docs.sciml.ai/NonlinearSolve/stable/native/solvers/#forcing_strategies).

Q4: what does <i>reuse_A_if_factorization</i> refer to? 

References: book by Tim Kelley on Non-Linear Equations, papers by e.g. Dembo-Eisenstat; 

### Packages for Krylov Subspace Solvers 

1. [KrylovKit.jl](https://jutho.github.io/KrylovKit.jl/stable/man/linear/)
2. [Krylov.jl](https://jso.dev/Krylov.jl/stable/) In particular [restarted GMRES](https://jso.dev/Krylov.jl/stable/solvers/unsymmetric/#GMRES)
3. [iterativesolvers](https://iterativesolvers.julialinearalgebra.org/dev/)

### GMRES Solvers 

1. restart value (and loss of superlinear converge due to restart)?
2. initial guess (take e.g. solution from previous Newton iteration, from previous water depth value or from coarser mesh)?
3. stopping criterium (how many GMRES iterations can be solved by implementing forcing terms?)?
4. compare GMRES with alternative Krylov subspace solver that exploit short term recurrences?   

### Scalar ILU($\tau$) Preconditioners 

How does ILU out of the box perform? Can the ILU decompositon of the Jacobian froozen over various Newton iterations.   

### AMG Solver 

Q1: how does the AMG solver converge? Currently AMG is used as a preconditioner for GMRES. Does AMG used as a solver instead, i.e., without the use of an outer Krylov iteration (i.e., without the use of an outer GMRES iteration) still converge. What is the asymptotic rate of convergence of this solver? How does this rate depend on factors such as the grid density (number of degrees of freedom), the amount of linear wave damping, the number of grid points per wavelength, the number of pre and post smoothing steps, and the type of cycle (V-cycle vs. W-cycle). 

Q2: suppose switching the smoother inside AMG off, i.e., setting pre and post smoothing steps both to zero (this amounts to using the coarse grid correction as a preconditioner, i.e., using multigrid deflation). Does the outer GMRES iteration still converge? If not, by how much is convergence delayed? Is switching the smoother off a feasible way to reduce overall computational cost? 

Q3: what is the computational cost of the AMG solver? How do the computational cost of the AMG set-up and solve phase compare? What factors influence the computational cost of the set-up phase? Suppose that the speed of coarsening is defined as the reduction of degrees of freedom from a level to the next coarser level? Can the  speed of coarsening accuracy be influenced? Can a fast and therefore inaccurate coarsening be compensated by an outer GMRES (or other Krylov) iteration? Is switching to a fast and inaccurate coarsening a feasible way to reduce overall computational cost? (cfr. literature on aggressive coarsening and multi-pass interpolation)

Q4: the implementation AlgebraicMultigrid.jl provides both a Ruge-Stueben and smoothed aggregation variant to algebraic multigrid. Which one did you use? How the variant you used compare with its counterpart. Does the smoothed aggregation approach allow to accelerate the coarsening and reduce the computational cost of the application of the preconditioner. 

Q5: Implementation of AMG. Is there a benefit to switching the AMG solver that the PETSc library provides. See petsc.jl.  

### Block Jacobi Preconditioners    

Assume a spatial discretization using $N_c$ cells (or elements or any alternative denoting the spatial accuracy) and using $N_h$ harmonics. Then the Jacobian is of size $N_c*N_h$. Martin showed the Jacobian as $N_c$ blocks, of size $N_h$. We instead would like to see the Jacobian as $N_h$ blocks of size $N_d$ (thus arriving at a smaller number of blocks that are large in size, especially in case of two-dimensional discretizations at high spatial resolution). 

1. preconditioners [BlockArrays](https://github.com/JuliaArrays/BlockArrays.jl)
2. [BlockFactorizations](https://github.com/SebastianAment/BlockFactorizations.jl) provides allocation-free matrix vector products;

### Parallel in Frequency using pmap 

Can the $N_h$ blocks of size $N_c$ be solved independently from each other using the function <i>pmap</i>? 

### Spatial Coarse Grid to Curb Increase in GMRES Iterations as $N_h$ Increases  

Can the increase in GMRES iterationss with $N_h$ bew curbed using coarser grids (lower spatial resolution).

## Section 2: 1D Shallow Water 

### Model Equations 

### Transient Analysis 

### Time-Harmonic Analysis 

## Section 3: 2D Shallow Water 

### Transient Analysis 

### Time-Harmonic Analysis 

## Section 4: Bifurcation Analysis 

Solve for gradually smaller height of the channel (as the non-linearity is proportional to inverse of height).  

## Section 5: Draft Paper 

* Tentative title: A Fast Newton - Krylov Solver for Multi-Harmonic Tidal Shallow Water Flow 
* Authors: MV, HS and DL 
* Scientific contribution: block preconditioner for Jacobian linear system. Blocks group degrees of freedom per harmonic component. Blocks can be approximately inverted using algebraic multigrid. Matrix-free Krylov implementation using  forward mode automatic differentiation.  Multi-threaded implementation using parallel processing in Julia programming language. 

### Section 1: Introduction 

<b>Problem description</b>: shallow water flow with periodic (tidal) forcing. Non-linear friction terms cause higher 
harmonics to arise.     

<b>State of the Art</b>: transient approach preferred as linear algebra for multi-harmonic considered to be computationally too expensive.   

<b>Scientific Contribution</b>: We propose a multi-harmonic solver in which the non-linear system for the coupled modes is solved by a Newton-Krylov method. Block preconditioner is performed  by grouping degrees of freedom corresponding to the various harmonic modes. Each block corresponds to a damped Helmholtz operator. These operators can efficiently be inverted using an algebraic multigrid method. 

<b>Numerical Results Obtained</b>: Quantify computational efficiency obtained.  

<b>Outside Scope</b>
* rectangular domains only;
* no morphodynamics; 

<b>Structure of Paper</b>: This paper is structured as follows. 

### Section 2: Tidal Shallow Water Flow

Depth-averaged shallow water flow with periodic forcing and non-linear friction terms. Rectangular computational domain, boundary conditions and initial conditions. Staggered finite difference discretization.  

### Section 3: Multi-Harmonic Formulation 

Truncated Fourier series. Harmonic balance method. Newton method for harmonic amplitudes. Jacobian computation. Resulting large linear systems with non-standard structure.    

### Section 4: Newton - Krylov - Multigrid Method  

Matrix-free formulation using forward-mode automatic (code) differentation. Block preconditioning. Damped Helmholtz system. Algebraic multigrid approximate solve. 

### Section 5: Numerical Results  

### Section 6: Conclusions 

### References 

## References 

Hierbij de link naar de files met het model:

https://filesender.surf.nl/?s=download&token=a7bd1605-4e56-485e-96e3-6d1750ba2572

Ik heb nog even gevraagd hoe de auteur (Haoyan) de niet-lineaire term doet. Hij doet inderdaad wat Martin voorstelde:

Mijn vraag: Do you substitute the velocity signal from the previous iteration and then do a FT, or do you have a 'closed' form solution?

Antwoord:

I followed the first approach: substituting the velocity signal from the previous iteration and then applying the Fourier transform.

As shown in Equation S.14, \phi^{j-1} indicates that it comes from the previous iteration.

Misschien kan dit slimmer (Domenico weet misschien een truc), maar deze werkt iig. Weet niet of het slim is om te discretiseren, ik denk dat het beter is een functie te maken (met de coefficienten uit de vorige iteratie), die te vermenigvuldigen met een specifieke mode (gebruik orthogonaliteit) en die dan met een slimme integrator te evalueren. Kunnen we het volgende week over hebben.

Merk trouwens ook op dat Haoyan niet vermenigvuldigd met (H+\zeta-h) maar die samen met de |u| numeriek evalueert zoals boven geschreven.